# Inter-model Label Agreement

Compares LLM labels across models (`data/CT-RATE_reports_{model}.csv`).  
All pairwise combinations are computed automatically — adding a new model CSV to `data/` is all that is needed.

In [ ]:
import itertools
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, cohen_kappa_score

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'figure.dpi': 150,
})

DATA_DIR    = Path('../data')
FIGURES_DIR = Path('../figures')
TABLES_DIR  = Path('../tables')

## 1. Load Models

In [ ]:
LABEL_COLS = [
    'Medical material',
    'Arterial wall calcification',
    'Cardiomegaly',
    'Pericardial effusion',
    'Coronary artery wall calcification',
    'Hiatal hernia',
    'Lymphadenopathy',
    'Emphysema',
    'Atelectasis',
    'Lung nodule',
    'Lung opacity',
    'Pulmonary fibrotic sequela',
    'Pleural effusion',
    'Mosaic attenuation pattern',
    'Peribronchial thickening',
    'Consolidation',
    'Bronchiectasis',
    'Interlobular septal thickening',
]

# Auto-detect model CSVs (exclude the reference file and any non-ASCII named copies)
candidate_csvs = sorted(DATA_DIR.glob('CT-RATE_reports_*.csv'))
model_csvs = [
    p for p in candidate_csvs
    if p.name.isascii()  # skip e.g. 'のコピー' artefacts
]

print('Model CSVs found:')
for p in model_csvs:
    print(' ', p.name)

# Load each model into a dict keyed by model name
model_dfs = {}   # {model_name: DataFrame with VolumeName + label cols renamed as col__model}
raw_dfs   = {}   # {model_name: original DataFrame, for prevalence queries}

for path in model_csvs:
    name = path.stem.replace('CT-RATE_reports_', '')
    df = pd.read_csv(path)
    n_total = len(df)
    if '_extraction_failed' in df.columns:
        df = df[~df['_extraction_failed'].fillna(True).astype(bool)].copy()
        print(f'{name}: removed {n_total - len(df):,} extraction failures')
    raw_dfs[name] = df
    model_dfs[name] = (
        df[['VolumeName'] + LABEL_COLS]
        .rename(columns={c: f'{c}__{name}' for c in LABEL_COLS})
    )
    print(f'Loaded {name}: {len(model_dfs[name]):,} rows')

# Merge all models on VolumeName (inner join — only cases present in all models)
merged = None
for name, df in model_dfs.items():
    merged = df if merged is None else merged.merge(df, on='VolumeName', how='inner')

# Add split info from reference
ref = pd.read_csv(DATA_DIR / 'CT-RATE_reports.csv')[['VolumeName', 'split']]
merged = merged.merge(ref, on='VolumeName', how='left')

model_names = list(model_dfs.keys())
pairs = list(itertools.combinations(model_names, 2))

print(f'\nMerged rows : {len(merged):,}')
print(f'Models      : {model_names}')
print(f'Pairs       : {[(a, b) for a, b in pairs]}')
print('\nSplit distribution:')
print(merged['split'].value_counts().to_string())

## 2. Pairwise Agreement Metrics

For each model pair and each label:
- **Prev_A / Prev_B** — prevalence (%) for each model
- **Overall agreement** — proportion of cases where both models agree
- **Positive agreement** — index of specific agreement for the positive category (Cicchetti & Feinstein 1990)
- **Negative agreement** — index of specific agreement for the negative category
- **Discordance rate** — proportion of cases where the two models disagree
- **Cohen's κ** — chance-corrected agreement

In [ ]:
def pairwise_metrics(df, model_a, model_b, label_cols=LABEL_COLS):
    rows = []
    for col in label_cols:
        ya = df[f'{col}__{model_a}'].astype(int).values
        yb = df[f'{col}__{model_b}'].astype(int).values
        n = len(ya)

        cm = confusion_matrix(ya, yb, labels=[0, 1])
        both_neg, aneg_bpos, apos_bneg, both_pos = cm.ravel()

        overall_agr = (both_pos + both_neg) / n
        denom_pos   = 2*both_pos + apos_bneg + aneg_bpos
        denom_neg   = 2*both_neg + apos_bneg + aneg_bpos
        pos_agr     = 2*both_pos / denom_pos if denom_pos > 0 else np.nan
        neg_agr     = 2*both_neg / denom_neg if denom_neg > 0 else np.nan
        disc        = (apos_bneg + aneg_bpos) / n

        try:
            kappa = cohen_kappa_score(ya, yb)
        except Exception:
            kappa = np.nan

        rows.append({
            'Pair':               f'{model_a} vs {model_b}',
            'Model_A':            model_a,
            'Model_B':            model_b,
            'Label':              col,
            'N':                  n,
            'Prev_A_%':           round(ya.mean() * 100, 1),
            'Prev_B_%':           round(yb.mean() * 100, 1),
            'BothPos':            int(both_pos),
            'Apos_Bneg':          int(apos_bneg),
            'Aneg_Bpos':          int(aneg_bpos),
            'BothNeg':            int(both_neg),
            'Overall_agreement':  round(overall_agr, 3),
            'Positive_agreement': round(pos_agr, 3) if not np.isnan(pos_agr) else np.nan,
            'Negative_agreement': round(neg_agr, 3) if not np.isnan(neg_agr) else np.nan,
            'Discordance_rate':   round(disc, 3),
            'Kappa':              round(kappa, 3) if not np.isnan(kappa) else np.nan,
        })
    return pd.DataFrame(rows)


# Compute all pairs and concatenate into one tidy DataFrame
all_metrics = pd.concat(
    [pairwise_metrics(merged, a, b) for a, b in pairs],
    ignore_index=True,
)

print('Table 1. Per-label pairwise agreement\n')
for pair_name, grp in all_metrics.groupby('Pair'):
    print(f'--- {pair_name} ---')
    show = grp.set_index('Label')[[
        'N', 'Prev_A_%', 'Prev_B_%',
        'Overall_agreement', 'Positive_agreement', 'Negative_agreement',
        'Discordance_rate', 'Kappa',
    ]]
    # Rename Prev columns to actual model names
    a, b = grp.iloc[0][['Model_A', 'Model_B']]
    show = show.rename(columns={'Prev_A_%': f'Prev_{a}_%', 'Prev_B_%': f'Prev_{b}_%'})
    print(show.to_string())
    print()

In [ ]:
# Macro-averaged summary across labels
agg_cols = ['Overall_agreement', 'Positive_agreement', 'Negative_agreement',
            'Discordance_rate', 'Kappa']
macro = (
    all_metrics.groupby('Pair')[agg_cols]
    .mean()
    .round(3)
)
print('Table 2. Macro-averaged metrics (across all 18 labels)\n')
print(macro.to_string())

## 3. Figures

### Cohen's κ heatmap per label

In [ ]:
# Pivot: labels × pairs, values = κ
kappa_pivot = all_metrics.pivot(index='Label', columns='Pair', values='Kappa')

# Sort by mean κ across pairs
kappa_pivot = kappa_pivot.loc[kappa_pivot.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(max(5, len(pairs) * 2.5 + 2), 7))
sns.heatmap(
    kappa_pivot.astype(float),
    annot=True, fmt='.2f', cmap='RdYlGn',
    vmin=0, vmax=1, linewidths=0.4, linecolor='white',
    annot_kws={'size': 9}, cbar_kws={'label': "Cohen's κ", 'shrink': 0.7},
    ax=ax,
)
ax.set_title("Inter-model Cohen's κ per label", pad=10)
ax.tick_params(axis='x', rotation=20, labelsize=9)
ax.tick_params(axis='y', rotation=0,  labelsize=8)
ax.set_xlabel('')
ax.set_ylabel('')

fig.tight_layout()
plt.savefig(FIGURES_DIR / 'iagr_fig1_kappa_heatmap.png', bbox_inches='tight', dpi=200)
plt.show()

### Prevalence comparison across models

In [ ]:
# Compute per-model prevalence directly from each model's data (on matched rows)
prev_rows = []
for name in model_names:
    for col in LABEL_COLS:
        val = merged[f'{col}__{name}'].astype(int).mean() * 100
        prev_rows.append({'Model': name, 'Label': col, 'Prevalence_%': round(val, 1)})
prev_df = pd.DataFrame(prev_rows)

# Sort labels by mean prevalence
label_order = (
    prev_df.groupby('Label')['Prevalence_%'].mean()
    .sort_values(ascending=False).index.tolist()
)
prev_df['Label'] = pd.Categorical(prev_df['Label'], categories=label_order, ordered=True)
prev_df = prev_df.sort_values('Label')

x = np.arange(len(LABEL_COLS))
n_models = len(model_names)
w = 0.7 / n_models
colours = plt.cm.tab10(np.linspace(0, 0.4, n_models))

fig, ax = plt.subplots(figsize=(13, 4))
for i, name in enumerate(model_names):
    vals = prev_df[prev_df['Model'] == name]['Prevalence_%'].values
    ax.bar(x + (i - (n_models-1)/2) * w, vals, width=w,
           label=name, color=colours[i], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(label_order, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Prevalence (%)')
ax.set_title('Label prevalence by model')
ax.legend()
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)

fig.tight_layout()
plt.savefig(FIGURES_DIR / 'iagr_fig2_prevalence.png', bbox_inches='tight', dpi=200)
plt.show()

### Positive vs negative agreement scatter

In [ ]:
pair_colours = plt.cm.tab10(np.linspace(0, 0.6, len(pairs)))

fig, ax = plt.subplots(figsize=(8, 6))

for (a, b), colour in zip(pairs, pair_colours):
    pair_name = f'{a} vs {b}'
    sub = all_metrics[all_metrics['Pair'] == pair_name]
    sc = ax.scatter(
        sub['Negative_agreement'], sub['Positive_agreement'],
        s=80, color=colour, alpha=0.8, label=pair_name,
        edgecolors='grey', linewidths=0.4,
    )
    # Label only outlier points (κ < 0.6)
    for _, row in sub[sub['Kappa'] < 0.6].iterrows():
        ax.annotate(
            row['Label'], (row['Negative_agreement'], row['Positive_agreement']),
            fontsize=7, ha='left', va='bottom',
            xytext=(4, 3), textcoords='offset points', color=colour,
        )

ax.axhline(0.8, color='grey', linestyle='--', linewidth=0.8, alpha=0.5)
ax.axvline(0.8, color='grey', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_xlabel('Negative agreement')
ax.set_ylabel('Positive agreement')
ax.set_xlim(-0.05, 1.1)
ax.set_ylim(-0.05, 1.1)
ax.set_title('Positive vs negative agreement per label')
ax.legend()

fig.tight_layout()
plt.savefig(FIGURES_DIR / 'iagr_fig3_pos_neg_scatter.png', bbox_inches='tight', dpi=200)
plt.show()

### Discordance rate per label

In [ ]:
disc_pivot = all_metrics.pivot(index='Label', columns='Pair', values='Discordance_rate')

# Sort by mean discordance descending (most discordant first)
disc_pivot = disc_pivot.loc[disc_pivot.mean(axis=1).sort_values(ascending=False).index]

x = np.arange(len(disc_pivot))
n_pairs = len(pairs)
w = 0.7 / n_pairs

fig, ax = plt.subplots(figsize=(13, 4))
for i, col in enumerate(disc_pivot.columns):
    ax.bar(x + (i - (n_pairs-1)/2) * w, disc_pivot[col].values,
           width=w, label=col, alpha=0.85, color=pair_colours[i])

ax.set_xticks(x)
ax.set_xticklabels(disc_pivot.index, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Discordance rate')
ax.set_title('Discordance rate per label')
ax.legend()
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)

fig.tight_layout()
plt.savefig(FIGURES_DIR / 'iagr_fig4_discordance.png', bbox_inches='tight', dpi=200)
plt.show()

### Agreement breakdown (stacked bar, per label, one panel per pair)

In [ ]:
colours_map = {
    'BothPos':  '#4CAF50',
    'BothNeg':  '#2196F3',
    'Apos_Bneg':'#FF9800',
    'Aneg_Bpos':'#F44336',
}

n_pairs = len(pairs)
fig, axes = plt.subplots(n_pairs, 1, figsize=(13, 4 * n_pairs), squeeze=False)

for ax, (a, b) in zip(axes[:, 0], pairs):
    pair_name = f'{a} vs {b}'
    sub = all_metrics[all_metrics['Pair'] == pair_name].set_index('Label')

    # Sort by BothPos descending
    sub = sub.sort_values('BothPos', ascending=False)
    n_total = sub[['BothPos', 'BothNeg', 'Apos_Bneg', 'Aneg_Bpos']].sum(axis=1)
    pct = sub[['BothPos', 'BothNeg', 'Apos_Bneg', 'Aneg_Bpos']].div(n_total, axis=0) * 100

    labels_legend = {
        'BothPos':   f'{a}+ / {b}+',
        'BothNeg':   f'{a}− / {b}−',
        'Apos_Bneg': f'{a}+ / {b}−',
        'Aneg_Bpos': f'{a}− / {b}+',
    }

    bottoms = np.zeros(len(pct))
    x = np.arange(len(pct))
    for key in ['BothPos', 'BothNeg', 'Apos_Bneg', 'Aneg_Bpos']:
        vals = pct[key].values
        ax.bar(x, vals, bottom=bottoms,
               color=colours_map[key], label=labels_legend[key], width=0.7)
        bottoms += vals

    ax.set_xticks(x)
    ax.set_xticklabels(pct.index, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Percentage (%)')
    ax.set_ylim(0, 105)
    ax.set_title(f'Agreement breakdown — {pair_name}')
    ax.legend(loc='upper right', fontsize=8)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4)
    ax.set_axisbelow(True)

fig.tight_layout()
plt.savefig(FIGURES_DIR / 'iagr_fig5_breakdown.png', bbox_inches='tight', dpi=200)
plt.show()

## 4. Summary Table

In [ ]:
pub_rows = []
for _, row in all_metrics.iterrows():
    a, b = row['Model_A'], row['Model_B']
    pub_rows.append({
        'Pair':                       row['Pair'],
        'Label':                      row['Label'],
        'N':                          row['N'],
        f'Prev {a} (%)':              row['Prev_A_%'],
        f'Prev {b} (%)':              row['Prev_B_%'],
        f'{a}+ / {b}+':               row['BothPos'],
        f'{a}+ / {b}−':               row['Apos_Bneg'],
        f'{a}− / {b}+':               row['Aneg_Bpos'],
        f'{a}− / {b}−':               row['BothNeg'],
        'Overall agreement':          f"{row['Overall_agreement']:.2f}",
        'Positive agreement':         f"{row['Positive_agreement']:.2f}" if not pd.isna(row['Positive_agreement']) else 'N/A',
        'Negative agreement':         f"{row['Negative_agreement']:.2f}" if not pd.isna(row['Negative_agreement']) else 'N/A',
        'Discordance rate':           f"{row['Discordance_rate']:.3f}",
        "Cohen's κ":                  f"{row['Kappa']:.2f}" if not pd.isna(row['Kappa']) else 'N/A',
    })

pub_table = pd.DataFrame(pub_rows)

print('Table 3. Full per-label pairwise agreement table\n')
for pair_name, grp in pub_table.groupby('Pair'):
    print(f'--- {pair_name} ---')
    print(grp.drop(columns='Pair').to_string(index=False))
    print()

pub_table.to_csv(TABLES_DIR / 'inter_model_agreement_table.csv', index=False)
print('Saved: inter_model_agreement_table.csv')